# **Regork Customer Churn Analysis Project**

## **Imports**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import sys
!{sys.executable} -m pip install xgboost
import xgboost

In [2]:
train_df = pd.read_csv("customer_retention_training.csv")
test_df = pd.read_csv("test_features.csv")

print("Training Data (first 5 rows):")
print(train_df.head())
print("\nTest Data (first 5 rows):")
print(test_df.head())   


Training Data (first 5 rows):
    Status  Gender SeniorCitizen Partner Dependents  Tenure        Contract  \
0  Current    Male            No      No         No      27        One year   
1  Current  Female            No     Yes        Yes      27  Month-to-month   
2     Left  Female            No      No         No      72  Month-to-month   
3  Current    Male            No     Yes         No      53  Month-to-month   
4  Current    Male            No     Yes         No      62  Month-to-month   

  PaperlessBilling            PaymentMethod  MonthlyCharges  ...  \
0              Yes  Credit card (automatic)           81.92  ...   
1              Yes         Electronic check          102.10  ...   
2               No             Mailed check          104.55  ...   
3              Yes         Electronic check           91.30  ...   
4              Yes             Mailed check           84.06  ...   

   NumInternationalCalls NumAppCrashes AppSessionLengthAvg NumPaymentFailures  \
0    

In [3]:
print("Training Shape:", train_df.shape)
print("Test Shape:", test_df.shape)
 

Training Shape: (6000, 35)
Test Shape: (1000, 34)


In [4]:
print("\nTraining Data Info:")
print(train_df.info())
print("\nTest Data Info:")
print(test_df.info())



Training Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 35 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Status                 6000 non-null   object 
 1   Gender                 6000 non-null   object 
 2   SeniorCitizen          6000 non-null   object 
 3   Partner                6000 non-null   object 
 4   Dependents             6000 non-null   object 
 5   Tenure                 6000 non-null   int64  
 6   Contract               6000 non-null   object 
 7   PaperlessBilling       6000 non-null   object 
 8   PaymentMethod          6000 non-null   object 
 9   MonthlyCharges         6000 non-null   float64
 10  TotalCharges           6000 non-null   float64
 11  PhoneService           6000 non-null   object 
 12  MultipleLines          6000 non-null   object 
 13  InternetService        6000 non-null   object 
 14  OnlineSecurity         6000 non-nul

In [5]:
print("\nTraining Data Description:")
print(train_df.describe())
print("\nTest Data Description:")
print(test_df.describe()) 


Training Data Description:
            Tenure  MonthlyCharges  TotalCharges  NumAppLoginsLastMonth  \
count  6000.000000     6000.000000   6000.000000            6000.000000   
mean     36.344500       78.512005   2850.679813              24.536667   
std      20.699313       21.716658   1856.929771              14.353745   
min       1.000000       18.250000     18.800000               0.000000   
25%      18.750000       70.277500   1282.900000              12.000000   
50%      36.000000       84.135000   2594.965000              24.000000   
75%      54.000000       93.952500   4303.395000              37.000000   
max      72.000000      118.750000   8067.130000              49.000000   

       EmailOpenRate  ReferralCount  NumPageViews  AvgCallDuration  \
count    6000.000000    6000.000000   6000.000000      6000.000000   
mean        0.200660       4.506833    100.059562         5.003643   
std         0.097978       2.884847     30.101693         2.000071   
min         0.00

In [6]:
# Clean columns
train_df.columns = train_df.columns.str.strip()
test_df.columns = test_df.columns.str.strip()

# Target Variable
train_df["Status"] = train_df["Status"].str.strip().map({"Current": 0, "Left": 1})

# Quick checks
print(train_df.isna().sum())
print(train_df["Status"].value_counts())
print("Churn rate:", train_df["Status"].mean())

# Column types
num_cols = train_df.drop(columns="Status").select_dtypes(include=["int64", "float64"]).columns
cat_cols = train_df.drop(columns="Status").select_dtypes(include=["object"]).columns


Status                      0
Gender                      0
SeniorCitizen               0
Partner                     0
Dependents                  0
Tenure                      0
Contract                    0
PaperlessBilling            0
PaymentMethod               0
MonthlyCharges              0
TotalCharges                0
PhoneService                0
MultipleLines               0
InternetService             0
OnlineSecurity              0
OnlineBackup                0
DeviceProtection            0
TechSupport                 0
StreamingTV                 0
StreamingMovies             0
NumAppLoginsLastMonth       0
EmailOpenRate               0
ReferralCount               0
NumPageViews                0
AvgCallDuration             0
NumInternationalCalls       0
NumAppCrashes               0
AppSessionLengthAvg         0
NumPaymentFailures          0
DataOverageCount            0
InternalCodeFlag            0
RandomIDHash                0
SurveyVersion               0
LegacySyst

## **Part 1: Exploratory Data Analysis**

**Question 1:**

In [7]:
churn_rate = train_df["Status"].mean()
print("Overall Churn Rate:", churn_rate)
print("Overall Churn Rate (%):", round(churn_rate * 100, 2))

Overall Churn Rate: 0.09916666666666667
Overall Churn Rate (%): 9.92


**Question 2:**

In [8]:
average_tenure = train_df.groupby("Status")["Tenure"].mean()

print("Mean Tenure for Existing Customers (Months):", average_tenure[0])
print("Mean Tenure for Churned Customers (Months):", average_tenure[1])

Mean Tenure for Existing Customers (Months): 36.5641073080481
Mean Tenure for Churned Customers (Months): 34.34957983193277


**Question 3:**

In [9]:
monthly_charges = train_df.groupby("Contract")["MonthlyCharges"].mean()

print("Mean Monthly Charges for Month-to-month:", monthly_charges["Month-to-month"])
print("Mean Monthly Charges for One year:", monthly_charges["One year"])

Mean Monthly Charges for Month-to-month: 81.5205493171472
Mean Monthly Charges for One year: 77.26054922973879


**Question 4:**

In [10]:
payment_churn = train_df.groupby("PaymentMethod")["Status"].mean()
print(payment_churn)

PaymentMethod
Bank transfer (automatic)    0.082584
Credit card (automatic)      0.085202
Electronic check             0.132057
Mailed check                 0.078032
Name: Status, dtype: float64


**Question 5:**

In [11]:
app_logins = train_df.groupby("Status")["NumAppLoginsLastMonth"].mean()

print("Mean app logins for Existing customers:", app_logins[0])
print("Mean app logins for Churned customers:", app_logins[1])


Mean app logins for Existing customers: 24.607030527289545
Mean app logins for Churned customers: 23.89747899159664


**Question 6:**

In [12]:
hash_churn = train_df.groupby("RandomIDHash")["Status"].mean()
print(hash_churn)

RandomIDHash
X1    0.093586
X2    0.106623
X3    0.102113
X4    0.094682
Name: Status, dtype: float64


Interpretation: these churn rates are all pretty close, so RandomIDHash does not look influential. It looks like noise, which makes sense because it is just a hashed ID, not customer behavior.

**Question 7:**

In [13]:
dependents_churn = train_df.groupby("Dependents")["Status"].mean()

print("Churn rate without dependents:", dependents_churn["No"])
print("Churn rate with dependents:", dependents_churn["Yes"])

Churn rate without dependents: 0.09846153846153846
Churn rate with dependents: 0.10084507042253521


**Question 8:**

In [14]:
email_open = train_df.groupby("Status")["EmailOpenRate"].mean()

print("Mean EmailOpenRate for Existing customers:", email_open[0])
print("Mean EmailOpenRate for Churned customers:", email_open[1])

Mean EmailOpenRate for Existing customers: 0.20111139958279373
Mean EmailOpenRate for Churned customers: 0.19656246217983192


**Question 9:**

In [15]:
contract_churn = train_df.groupby("Contract")["Status"].mean()

print("Month-to-month churn rate:", contract_churn["Month-to-month"])
print("One year churn rate:", contract_churn["One year"])
print("Two year churn rate:", contract_churn["Two year"])

Month-to-month churn rate: 0.16024279210925646
One year churn rate: 0.022103148024112524
Two year churn rate: 0.028052805280528052


**Question 10:**

In [16]:
train_df["TenureGroup"] = pd.cut(
    train_df["Tenure"],
    bins=[0, 12, 24, float("inf")],
    labels=["< 12 months", "12-24 months", "> 24 months"],
    right=False
)

q10_tenure_group_churn = train_df.groupby("TenureGroup")["Status"].mean()
print(q10_tenure_group_churn)

TenureGroup
< 12 months     0.104626
12-24 months    0.106549
> 24 months     0.096092
Name: Status, dtype: float64


C:\Users\fwkep\AppData\Local\Temp\ipykernel_14904\3751829047.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  q10_tenure_group_churn = train_df.groupby("TenureGroup")["Status"].mean()


**Question 11:**

In [17]:
categorical_cols = train_df.select_dtypes(include="object").columns

q11_results = []

for col in categorical_cols:
    churn_rates = train_df.groupby(col)["Status"].mean()
    churn_diff = churn_rates.max() - churn_rates.min()
    
    q11_results.append({
        "Variable": col,
        "ChurnRateDifference": churn_diff
    })

q11_df = pd.DataFrame(q11_results).sort_values(
    by="ChurnRateDifference", ascending=False
)

print(q11_df.head(5))

           Variable  ChurnRateDifference
4          Contract             0.138140
9   InternetService             0.121428
15  StreamingMovies             0.088821
14      StreamingTV             0.086224
13      TechSupport             0.085021


**Question 12:**

In [18]:
numeric_cols = train_df.select_dtypes(include=["int64", "float64"]).columns
numeric_cols = numeric_cols.drop("Status")

q12_results = []

for col in numeric_cols:
    mean_stay = train_df.groupby("Status")[col].mean()[0]
    mean_churn = train_df.groupby("Status")[col].mean()[1]
    
    relative_diff = abs(mean_churn - mean_stay) / mean_stay
    
    q12_results.append({
        "Variable": col,
        "Mean_Stay": mean_stay,
        "Mean_Churn": mean_churn,
        "Relative_Difference": relative_diff
    })

q12_df = pd.DataFrame(q12_results).sort_values(
    by="Relative_Difference", ascending=False
)

print(q12_df.head(5))


            Variable    Mean_Stay   Mean_Churn  Relative_Difference
1     MonthlyCharges    77.318477    89.354050             0.155662
2       TotalCharges  2823.555920  3097.074168             0.096870
0             Tenure    36.564107    34.349580             0.060566
12  DataOverageCount     1.503608     1.450420             0.035373
5      ReferralCount     4.521739     4.371429             0.033242


**Question 13:**

In [19]:
missing = train_df.isna().sum()
missing = missing[missing > 0]
print(missing)

PromoCategory    1469
dtype: int64


**Question 14:**

Noisy features do not contain any information about customer churn in this case, so they distract the model from data that matters. Distracting the model can result in a loss of prediction accuracy, therefore making the model less reliable. A good example is the RandomIDHash that we previously analyzed. 

**Question 15:**

One feature I believe to be signal is Contract. The Churn rate for month-to-month customrs is 16.02% compared with only 2.21% for one-year customers and 2.81% for two-year customers. This is a very large difference in churn rate, which suggests that contract type is strongly related to whether a customer leaves. Therefore, customers without a longer commitment appear much more likely to churn.

One feature I believe to be noise is RandomIDHash that I was just referring to. The churn rates for the hash groups are all very close, varying from 9.36% to 10.66%. Since this variable is used as just an identifier of customers, I don't believe it offers and information that we can quantify when referring to churn. 

**Question 16:**

Based on my EDA, the two features with the strongest influence on churn are Contract and Monthly Charges. Contract type showed the clearest difference in churn rates. The Churn rate for month-to-month customrs is 16.02% compared with only 2.21% for one-year customers and 2.81% for two-year customers. This is a very large difference in churn rate, which suggests that contract type is strongly related to whether a customer leaves. Therefore, customers without a longer commitment appear much more likely to churn.

MonthlyCharges also appears to be an important driver to churn. Customers who churned had average monthly charges of about 89.35, compared with 77.32 for customers who stayed. That is roughly a 15.6% higher monthly bill among churned customers. This suggests that price sensitivity may be pushing some customers to leave, especially if they do not feel they are getting enough value for what they pay.

A business recommendation would be to target high-bill, month-to-month customers with retention offers. Regork Telecom could offer discounts for moving into annual plans, provide bundling options, or reach out with lower-cost plan options before those customers decide to leave.

**Question 17:**

One limitation of this dataset is that it does not include a detailed timeline of customer behavior before churn. We can see values such as app logins, email open rate, and payment failures, but we do not know exactly when those behaviors changed or how close they were to the customer leaving. A second limitation is that we do not have customer feedback measures such as complaint history or satisfaction ratings. Those factors could strongly influence churn but are not captured here.

From our EDA we can see different patterns and relationships in our target variable, but it does not prove causation. For example, month-to-month customers had much higher churn, but that does not mean the contract type alone caused them to leave. Other related factors such as pricing, service quality, or support issues may also be contributing.

If I had more time, the next step I would take would be to build a predictive model and test whether the strongest EDA variables, especially contract type and monthly charges, still matter after accounting for the other features. That would help confirm which patterns are most useful for churn prediction.

## **Part 2: Structured Predictive Modeling**

### **Step 1: Setup**

In [20]:
model_df = pd.read_csv("customer_retention_training.csv")
model_df.columns = model_df.columns.str.strip()

X = model_df.drop(columns="Status")
y = model_df["Status"].str.strip().map({"Current": 0, "Left": 1})

print("X shape:", X.shape)
print("y shape:", y.shape)
print(y.value_counts())

X shape: (6000, 34)
y shape: (6000,)
Status
0    5405
1     595
Name: count, dtype: int64


**Question 18:**

We encode the target variable as binary because MLMs work with predicting numerical outcomes, not text.

### **Step 2: Train/Test Split**

**Question 19 & 20:**

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (4200, 34)
X_test shape: (1800, 34)
y_train shape: (4200,)
y_test shape: (1800,)


**Question 21:**

We use stratify=y because churn is not evenly split between the two classes. In this specific example, most customers are Current (0), so the dataset is not close to being even. Stratification keeps the proportion of churned and non-churned customers similar in both the training and test sets.

### **Step 3: Preprocesing Pipeline**

In [22]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X_train.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ])

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 13
Categorical features: 21


**Question 22:**

The main benefit of a preprocessing pipeline is that it applies the same methodology to both training and test datasets. This makes our model more reproducible, helps prevent leakage, and gets rid of any inconsistencies from manual processing. 

### **Step 4: Cross-Validation**

In [23]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("StratifiedKFold created with 5 splits, shuffle=True, random_state=42")

StratifiedKFold created with 5 splits, shuffle=True, random_state=42


**Question 23:**

We use cross-validation instead of a single train/test split because it evaluates the model across multiple different subsets of the training data. It also reduces the chance that our results depend too much on one lucky/unlucky split. Because of this cross-validation gives you a much more reliable estimate of model performance.

### **Step 5: Model Training & Tuning**

In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

**Model A: Logistic Regression (Question 24)**

In [25]:
log_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

log_param_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs"]
}

log_grid = GridSearchCV(
    estimator=log_pipe,
    param_grid=log_param_grid,
    scoring="f1",
    cv=cv
)

log_grid.fit(X_train, y_train)

print("Best Logistic Params:", log_grid.best_params_)
print("Best Logistic CV F1:", log_grid.best_score_)

Best Logistic Params: {'model__C': 10, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
Best Logistic CV F1: 0.16904118683353864


**Model B: Decision Tree (Question 25)**

In [26]:
tree_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])

tree_param_grid = {
    "model__max_depth": [5, 10],
    "model__min_samples_split": [2, 10],
    "model__min_samples_leaf": [1, 5]
}

tree_grid = GridSearchCV(
    estimator=tree_pipe,
    param_grid=tree_param_grid,
    scoring="f1",
    cv=cv
)

tree_grid.fit(X_train, y_train)

print("Best Tree Params:", tree_grid.best_params_)
print("Best Tree CV F1:", tree_grid.best_score_)


Best Tree Params: {'model__max_depth': 10, 'model__min_samples_leaf': 5, 'model__min_samples_split': 2}
Best Tree CV F1: 0.2002625004953164


**Model C: Random Forest (Question 26)**

In [27]:
rf_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

rf_param_grid = {
    "model__n_estimators": [200, 500],
    "model__max_depth": [5, 10, 20],
    "model__min_samples_split": [2, 5]
}

rf_grid = GridSearchCV(
    estimator=rf_pipe,
    param_grid=rf_param_grid,
    scoring="f1",
    cv=cv
)

rf_grid.fit(X_train, y_train)

print("Best RF Params:", rf_grid.best_params_)
print("Best RF CV F1:", rf_grid.best_score_)

Best RF Params: {'model__max_depth': 20, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Best RF CV F1: 0.014064230343300111


### **Step 6: Model Selection**

**Question 27:**

In [28]:
if log_grid.best_score_ >= tree_grid.best_score_ and log_grid.best_score_ >= rf_grid.best_score_:
    best_model = log_grid
    best_model_name = "Logistic Regression"
elif tree_grid.best_score_ >= rf_grid.best_score_:
    best_model = tree_grid
    best_model_name = "Decision Tree"
else:
    best_model = rf_grid
    best_model_name = "Random Forest"

print("Best model:", best_model_name)
print("Best CV F1:", best_model.best_score_)

Best model: Decision Tree
Best CV F1: 0.2002625004953164


### **Step 7: Feature Importance**

In [29]:
from sklearn.inspection import permutation_importance

**Question 28:**

In [30]:
best_estimator = best_model.best_estimator_

perm_result = permutation_importance(
    best_estimator,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="f1"
)

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance_Mean": perm_result.importances_mean,
    "Importance_STD": perm_result.importances_std
}).sort_values(by="Importance_Mean", ascending=False)

print(importance_df.head(5))

               Feature  Importance_Mean  Importance_STD
12     InternetService         0.045628        0.014598
5             Contract         0.041042        0.020462
27  NumPaymentFailures         0.016383        0.007320
7        PaymentMethod         0.014712        0.017223
30        RandomIDHash         0.006845        0.003656


### **Step 8: Evaluation**

**Question 29, 30, 31, & 32:**

In [31]:
y_pred = best_estimator.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred)
test_recall = recall_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred)

print("Accuracy:", test_accuracy)
print("Precision:", test_precision)
print("Recall:", test_recall)
print("F1 Score:", test_f1)

Accuracy: 0.8694444444444445
Precision: 0.1724137931034483
Recall: 0.08426966292134831
F1 Score: 0.11320754716981132


### **Step 9: Interpretation**

In [32]:
cv_f1 = best_model.best_score_
test_f1 = test_f1

difference = cv_f1 - test_f1

print("Cross-Validation F1:", cv_f1)
print("Test F1:", test_f1)
print("Difference:", difference)


Cross-Validation F1: 0.2002625004953164
Test F1: 0.11320754716981132
Difference: 0.08705495332550509


**Question 33:**

The best model’s cross-validation F1 score was 0.200, and its test set F1 score was 0.113. The difference betwween the two of them is 0.0871. This  suggests overfitting, especially because the test set F1 score is just over half of the CV F1 score. This model dropped on the test set by a considerable amount, so it may not have the ability to generalize new data/customers very well.

**Question 34:**

Generalization means how well a model performs on new data it has never seen before. A model that genralizes well can find patterns that are seen in the real world. 

The model's generalization appears to be limited. The drop from the CV model to the Test set suggests that the model does not perform very well on new data, relative to how it did during training and validation trials. 

One concern about future performance is that the dataset contains some noisy or weak variables, such as the IDs and internal flags. If the model relies on any patterns from those columns, its predictions become less reliable when applied to new data. 

**Question 35:**

The top two features that were identified by permutation importance are **Internet Service** and **Contract**, respectively. These features have the strongest influence on whether a customer is predicted to churn in our current model. 

In business terms, the **Internet Service** feature represents the type of internet product the customer has, such as fiber optic, DSL, or no internet service. This matters because different service types could create different customer experiences and pricing levels. If one service type is associated with more churn that suggests some customers may be dissatisfied with the value or quality that they are receiving. The **Contract** feature represents how long the plan of the customer is on. Customers on the month-to-month plan can leave more easily, and are not as "locked in" as the other customers on plans. Customers on longer-term contracts are more likely to stay over the contract period. 

A recommendation for Regork would be to focus on retention efforts on **customers with higher-risk internet service types** and/or **customers who are on the month-to-month contracts**. The company could offer incentives to move month-to-month customers into longer-term contracts and do service quality reviews/outreach. 

**Question 36:**

In [33]:
models = {
    "Logistic Regression": log_grid.best_estimator_,
    "Decision Tree": tree_grid.best_estimator_,
    "Random Forest": rf_grid.best_estimator_
}

comparison_results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    
    comparison_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "CV_F1": {
            "Logistic Regression": log_grid.best_score_,
            "Decision Tree": tree_grid.best_score_,
            "Random Forest": rf_grid.best_score_
        }[name]
    })

comparison_df = pd.DataFrame(comparison_results)
print(comparison_df.sort_values(by="F1", ascending=False))


                 Model  Accuracy  Precision    Recall        F1     CV_F1
1        Decision Tree  0.869444   0.172414  0.084270  0.113208  0.200263
0  Logistic Regression  0.900556   0.466667  0.039326  0.072539  0.169041
2        Random Forest  0.901111   0.500000  0.005618  0.011111  0.014064


I recommend that Regork use the **Decision Tree** model. Among the three models, it achieved the highest CV F1 score of 0.200, highest test F1 score of 0.113, highest recall of 0.0843, and had the lowest precision of 0.1724. These results are not the strongest, but they are still the best compared to the Logistic Regression and Random Forest models. Since predicting churn is about identifying customers who are likely to leave, F1 is more useful than accuracy alone because it balances precision and recall. For Regork, recall is especially important because missing churners means losing customers who might have been saved. At the same time, precision still matters because the company does not want to waste retention efforts on too many customers who were unlikely to leave in the first place. 

The Decision Tree model also had the lowest difference between CV F1 score and the test F1 score. This represents that the Decision Tree model overfits the least out of the three of them.

Overall, I recommend the **Decision Tree** model because it delivered the best balance of performance and reliability compared to the other two models, making it our best choice so far. 


# **Part 3: Open-Ended Modeling Competition**

### **Reload Data**

In [34]:
train_full = pd.read_csv("customer_retention_training.csv")
test_submit = pd.read_csv("test_features.csv")

train_full.columns = train_full.columns.str.strip()
test_submit.columns = test_submit.columns.str.strip()

y = train_full["Status"].str.strip().map({"Current": 0, "Left": 1})
X_full = train_full.drop(columns="Status")

print("Training shape:", train_full.shape)
print("Feature shape:", X_full.shape)
print("Submission test shape:", test_submit.shape)
print(y.value_counts())

Training shape: (6000, 35)
Feature shape: (6000, 34)
Submission test shape: (1000, 34)
Status
0    5405
1     595
Name: count, dtype: int64


### **Refine Data**

In [35]:
# Building a reusable function for ROC AUC scoring
from sklearn.model_selection import cross_val_score

cv_auc = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def build_preprocessor(X):
    numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
    categorical_features = X.select_dtypes(include=["object", "category"]).columns

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)
        ]
    )
    return preprocessor

def score_auc(X, y):
    model = Pipeline(steps=[
        ("preprocessor", build_preprocessor(X)),
        ("model", LogisticRegression(max_iter=1000))
    ])
    
    scores = cross_val_score(model, X, y, cv=cv_auc, scoring="roc_auc")
    return scores.mean()


In [36]:
# Testing noisy columns
drop_candidates = [
    "RandomIDHash",
    "InternalCodeFlag",
    "SurveyVersion",
    "LegacySystemScore",
    "PromoCategory"
]

results = []
baseline_auc = score_auc(X_full, y)

results.append({
    "Version": "Baseline (all features)",
    "ROC_AUC": baseline_auc
})

for col in drop_candidates:
    X_test_drop = X_full.drop(columns=[col])
    auc_score = score_auc(X_test_drop, y)
    
    results.append({
        "Version": f"Drop {col}",
        "ROC_AUC": auc_score
    })

drop_test_df = pd.DataFrame(results).sort_values(by="ROC_AUC", ascending=False)
print(drop_test_df)

                   Version   ROC_AUC
1        Drop RandomIDHash  0.796070
0  Baseline (all features)  0.795426
4   Drop LegacySystemScore  0.795297
5       Drop PromoCategory  0.794989
2    Drop InternalCodeFlag  0.794941
3       Drop SurveyVersion  0.794417


In [37]:
# Adding potential engineered features
def add_engineered_features(df):
    df = df.copy()
    
    df["IsMonthToMonth"] = (df["Contract"] == "Month-to-month").astype(int)
    df["IsAutoPay"] = df["PaymentMethod"].isin([
        "Bank transfer (automatic)",
        "Credit card (automatic)"
    ]).astype(int)
    
    service_cols = [
        "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
        "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
    ]
    df["ServiceCount"] = sum(df[col].eq("Yes").astype(int) for col in service_cols)
    
    df["SupportRiskScore"] = (
        df["NumPaymentFailures"] +
        df["NumAppCrashes"] +
        df["DataOverageCount"]
    )
    
    df["TenureGroup_P3"] = pd.cut(
        df["Tenure"],
        bins=[0, 12, 24, float("inf")],
        labels=["<12", "12-24", ">24"],
        right=False
    )
    
    return df

In [38]:
# Evaluation of engineered features relative to baseline
X_eng = add_engineered_features(X_full)

print("Baseline ROC AUC:", score_auc(X_full, y))
print("Engineered ROC AUC:", score_auc(X_eng, y))

Baseline ROC AUC: 0.7954259594679686
Engineered ROC AUC: 0.7939800527056337


Initial refine-data testing showed that dropping RandomIDHash slightly improved ROC AUC, while dropping the other questionable variables individually reduced performance. A first bundle of engineered features also reduced ROC AUC. Based on these results, the next round of Refining data testing will compare a conservative cleaned dataset against a more aggressive noise-removal version and then test engineered features more selectively.

In [39]:
drop_conservative = ["RandomIDHash"]
drop_aggressive = ["RandomIDHash", "InternalCodeFlag"]
drop_more_aggressive = [
    "RandomIDHash", "InternalCodeFlag", "SurveyVersion", "LegacySystemScore", "PromoCategory"]

X_cons = X_full.drop(columns=drop_conservative)
X_aggr = X_full.drop(columns=drop_aggressive)
X_more_aggr = X_full.drop(columns=drop_more_aggressive)

X_submit_cons = test_submit.drop(columns=drop_conservative)
X_submit_aggr = test_submit.drop(columns=drop_aggressive)
X_submit_more_aggr = test_submit.drop(columns=drop_more_aggressive)

print("Conservative shape:", X_cons.shape)
print("Aggressive shape:", X_aggr.shape)
print("More aggressive shape:", X_more_aggr.shape)

print("Conservative ROC AUC:", score_auc(X_cons, y))
print("Aggressive ROC AUC:", score_auc(X_aggr, y))
print("More aggressive ROC AUC:", score_auc(X_more_aggr, y))



Conservative shape: (6000, 33)
Aggressive shape: (6000, 32)
More aggressive shape: (6000, 29)
Conservative ROC AUC: 0.7960696211879756
Aggressive ROC AUC: 0.7961302559876865
More aggressive ROC AUC: 0.7948849104859336


It seems that the drop_aggressive is our best dataset, so we will drop RandomIDHash and InternalCodeFlag and keep the other three potentially noisy features for now. 

**Testing Engineered features:**

In [40]:
X_base = X_aggr.copy()
X_submit_base = X_submit_aggr.copy()

def add_feature_set(df, feature_name):
    df = df.copy()

    if feature_name == "IsMonthToMonth":
        df["IsMonthToMonth"] = (df["Contract"] == "Month-to-month").astype(int)

    elif feature_name == "IsAutoPay":
        df["IsAutoPay"] = df["PaymentMethod"].isin([
            "Bank transfer (automatic)",
            "Credit card (automatic)"
        ]).astype(int)

    elif feature_name == "ServiceCount":
        service_cols = [
            "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
            "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
        ]
        df["ServiceCount"] = sum(df[col].eq("Yes").astype(int) for col in service_cols)

    elif feature_name == "SupportRiskScore":
        df["SupportRiskScore"] = (
            df["NumPaymentFailures"] +
            df["NumAppCrashes"] +
            df["DataOverageCount"]
        )

    elif feature_name == "AvgChargePerMonth":
        df["AvgChargePerMonth"] = df["TotalCharges"] / df["Tenure"].replace(0, 1)

    return df

feature_candidates = [
    "IsMonthToMonth",
    "IsAutoPay",
    "ServiceCount",
    "SupportRiskScore",
    "AvgChargePerMonth"
]

feature_results = []

base_auc = score_auc(X_base, y)
feature_results.append({
    "Version": "Base (drop RandomIDHash + InternalCodeFlag)",
    "ROC_AUC": base_auc
})

for feat in feature_candidates:
    X_temp = add_feature_set(X_base, feat)
    auc_score = score_auc(X_temp, y)

    feature_results.append({
        "Version": f"Base + {feat}",
        "ROC_AUC": auc_score
    })

feature_results_df = pd.DataFrame(feature_results).sort_values(by="ROC_AUC", ascending=False)
print(feature_results_df)


                                       Version   ROC_AUC
1                        Base + IsMonthToMonth  0.796255
2                             Base + IsAutoPay  0.796208
0  Base (drop RandomIDHash + InternalCodeFlag)  0.796130
3                          Base + ServiceCount  0.796127
4                      Base + SupportRiskScore  0.796065
5                     Base + AvgChargePerMonth  0.795925


From this it seems that our best option is the Base + IsMonthToMonth. Lets analyze this with the addition of some engineered features from the InternetService column since it had a high permutation importance in Part 2. 

In [41]:
# New best base: aggressive drop set + best engineered feature so far
X_best = add_feature_set(X_aggr, "IsMonthToMonth")

def add_internet_feature(df, feature_name):
    df = df.copy()

    if feature_name == "IsFiberOptic":
        df["IsFiberOptic"] = (df["InternetService"] == "Fiber optic").astype(int)

    elif feature_name == "HasInternet":
        df["HasInternet"] = (df["InternetService"] != "No").astype(int)

    elif feature_name == "FiberMonthToMonth":
        df["FiberMonthToMonth"] = (
            (df["InternetService"] == "Fiber optic") &
            (df["Contract"] == "Month-to-month")
        ).astype(int)

    elif feature_name == "FiberNoTechSupport":
        df["FiberNoTechSupport"] = (
            (df["InternetService"] == "Fiber optic") &
            (df["TechSupport"] == "No")
        ).astype(int)

    return df

internet_candidates = [
    "IsFiberOptic",
    "HasInternet",
    "FiberMonthToMonth",
    "FiberNoTechSupport"
]

internet_results = []

base_auc = score_auc(X_best, y)
internet_results.append({
    "Version": "Base + IsMonthToMonth",
    "ROC_AUC": base_auc
})

for feat in internet_candidates:
    X_temp = add_internet_feature(X_best, feat)
    auc_score = score_auc(X_temp, y)

    internet_results.append({
        "Version": f"Base + IsMonthToMonth + {feat}",
        "ROC_AUC": auc_score
    })

internet_results_df = pd.DataFrame(internet_results).sort_values(by="ROC_AUC", ascending=False)
print(internet_results_df)


                                      Version   ROC_AUC
4  Base + IsMonthToMonth + FiberNoTechSupport  0.796623
0                       Base + IsMonthToMonth  0.796255
2         Base + IsMonthToMonth + HasInternet  0.796149
1        Base + IsMonthToMonth + IsFiberOptic  0.796105
3   Base + IsMonthToMonth + FiberMonthToMonth  0.795569


Right now our model looks like we will be dropping the RandomIDHash and InternalCodeFlag and adding IsMonthToMonth and FiberNoTechSupport. Next, I am going to analyze the IsAutoPay with the model we have now since that recorded better than our base. 

In [42]:
# Current best dataset
X_best2 = add_feature_set(X_aggr, "IsMonthToMonth")
X_best2 = add_internet_feature(X_best2, "FiberNoTechSupport")

def add_extra_feature(df, feature_name):
    df = df.copy()

    if feature_name == "IsAutoPay":
        df["IsAutoPay"] = df["PaymentMethod"].isin([
            "Bank transfer (automatic)",
            "Credit card (automatic)"
        ]).astype(int)

    elif feature_name == "ServiceCount":
        service_cols = [
            "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
            "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
        ]
        df["ServiceCount"] = sum(df[col].eq("Yes").astype(int) for col in service_cols)

    elif feature_name == "SupportRiskScore":
        df["SupportRiskScore"] = (
            df["NumPaymentFailures"] +
            df["NumAppCrashes"] +
            df["DataOverageCount"]
        )

    elif feature_name == "AvgChargePerMonth":
        df["AvgChargePerMonth"] = df["TotalCharges"] / df["Tenure"].replace(0, 1)

    return df

extra_candidates = ["IsAutoPay", "ServiceCount", "SupportRiskScore", "AvgChargePerMonth"]

extra_results = []
extra_results.append({
    "Version": "Current Best",
    "ROC_AUC": score_auc(X_best2, y)
})

for feat in extra_candidates:
    X_temp = add_extra_feature(X_best2, feat)
    extra_results.append({
        "Version": f"Current Best + {feat}",
        "ROC_AUC": score_auc(X_temp, y)
    })

extra_results_df = pd.DataFrame(extra_results).sort_values(by="ROC_AUC", ascending=False)
print(extra_results_df)


                            Version   ROC_AUC
0                      Current Best  0.796623
1          Current Best + IsAutoPay  0.796597
2       Current Best + ServiceCount  0.796575
3   Current Best + SupportRiskScore  0.796550
4  Current Best + AvgChargePerMonth  0.796426


**Final Refined Dataset:**

In [43]:
X_part3 = X_aggr.copy()
X_submit_part3 = X_submit_aggr.copy()

X_part3["IsMonthToMonth"] = (X_part3["Contract"] == "Month-to-month").astype(int)
X_submit_part3["IsMonthToMonth"] = (X_submit_part3["Contract"] == "Month-to-month").astype(int)

X_part3["FiberNoTechSupport"] = (
    (X_part3["InternetService"] == "Fiber optic") &
    (X_part3["TechSupport"] == "No")
).astype(int)

X_submit_part3["FiberNoTechSupport"] = (
    (X_submit_part3["InternetService"] == "Fiber optic") &
    (X_submit_part3["TechSupport"] == "No")
).astype(int)

print("Final Part 3 feature shape:", X_part3.shape)
print("Current best ROC AUC:", score_auc(X_part3, y))


Final Part 3 feature shape: (6000, 34)
Current best ROC AUC: 0.7966231080776437


### **Train & Tune**

In [44]:
# Final Part 3 feature set based on refine-data results
X_part3 = X_aggr.copy()
X_submit_part3 = X_submit_aggr.copy()

# Best engineered features so far
X_part3["IsMonthToMonth"] = (X_part3["Contract"] == "Month-to-month").astype(int)
X_submit_part3["IsMonthToMonth"] = (X_submit_part3["Contract"] == "Month-to-month").astype(int)

X_part3["FiberNoTechSupport"] = (
    (X_part3["InternetService"] == "Fiber optic") &
    (X_part3["TechSupport"] == "No")
).astype(int)

X_submit_part3["FiberNoTechSupport"] = (
    (X_submit_part3["InternetService"] == "Fiber optic") &
    (X_submit_part3["TechSupport"] == "No")
).astype(int)

print("Final Part 3 training shape:", X_part3.shape)
print("Final Part 3 submission shape:", X_submit_part3.shape)
print("Current best ROC AUC:", score_auc(X_part3, y))


Final Part 3 training shape: (6000, 34)
Final Part 3 submission shape: (1000, 34)
Current best ROC AUC: 0.7966231080776437


In [45]:
from sklearn.model_selection import RandomizedSearchCV

In [46]:
cv_part3 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

numeric_features_part3 = X_part3.select_dtypes(include=["int64", "float64"]).columns
categorical_features_part3 = X_part3.select_dtypes(include=["object", "category"]).columns

numeric_transformer_part3 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_transformer_part3 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_part3 = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_part3, numeric_features_part3),
        ("cat", categorical_transformer_part3, categorical_features_part3)
    ]
)

print("Numeric features:", len(numeric_features_part3))
print("Categorical features:", len(categorical_features_part3))

Numeric features: 15
Categorical features: 19


**Building Models:**

In [47]:
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier

**Logistic Regression Model:**

In [48]:
log_part3_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor_part3),
    ("model", LogisticRegression(max_iter=1000))
])

log_part3_param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 50, 100],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs"]
}

log_part3_search = RandomizedSearchCV(
    estimator=log_part3_pipe,
    param_distributions=log_part3_param_grid,
    n_iter=6,
    scoring="roc_auc",
    cv=cv_part3,
    random_state=42,
    n_jobs=-1
)

log_part3_search.fit(X_part3, y)

print("Best Logistic Regression Params:", log_part3_search.best_params_)
print("Best Logistic Regression ROC AUC:", log_part3_search.best_score_)


Best Logistic Regression Params: {'model__solver': 'lbfgs', 'model__penalty': 'l2', 'model__C': 1}
Best Logistic Regression ROC AUC: 0.7966231080776437


**Decision Tree Model:**

In [49]:
tree_part3_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor_part3),
    ("model", DecisionTreeClassifier(random_state=42))
])

tree_part3_param_grid = {
    "model__max_depth": [3, 5, 10, 15, 20, None],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__criterion": ["gini", "entropy"]
}

tree_part3_search = RandomizedSearchCV(
    estimator=tree_part3_pipe,
    param_distributions=tree_part3_param_grid,
    n_iter=20,
    scoring="roc_auc",
    cv=cv_part3,
    random_state=42,
    n_jobs=-1
)

tree_part3_search.fit(X_part3, y)

print("Best Decision Tree Params:", tree_part3_search.best_params_)
print("Best Decision Tree ROC AUC:", tree_part3_search.best_score_)


Best Decision Tree Params: {'model__min_samples_split': 20, 'model__min_samples_leaf': 10, 'model__max_depth': 3, 'model__criterion': 'entropy'}
Best Decision Tree ROC AUC: 0.7809552313062135


**Random Forest Model:**

In [50]:
rf_part3_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor_part3),
    ("model", RandomForestClassifier(random_state=42))
])

rf_part3_param_grid = {
    "model__n_estimators": [200, 300, 500, 700],
    "model__max_depth": [5, 10, 15, 20, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 5],
    "model__max_features": ["sqrt", "log2", None]
}

rf_part3_search = RandomizedSearchCV(
    estimator=rf_part3_pipe,
    param_distributions=rf_part3_param_grid,
    n_iter=20,
    scoring="roc_auc",
    cv=cv_part3,
    random_state=42,
    n_jobs=-1
)

rf_part3_search.fit(X_part3, y)

print("Best Random Forest Params:", rf_part3_search.best_params_)
print("Best Random Forest ROC AUC:", rf_part3_search.best_score_)


Best Random Forest Params: {'model__n_estimators': 700, 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__max_features': 'log2', 'model__max_depth': 5}
Best Random Forest ROC AUC: 0.7895506028498356


**Extra Trees Model:**

In [51]:
et_part3_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor_part3),
    ("model", ExtraTreesClassifier(random_state=42))
])

et_part3_param_grid = {
    "model__n_estimators": [200, 300, 500, 700],
    "model__max_depth": [5, 10, 15, 20, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 5],
    "model__max_features": ["sqrt", "log2", None]
}

et_part3_search = RandomizedSearchCV(
    estimator=et_part3_pipe,
    param_distributions=et_part3_param_grid,
    n_iter=20,
    scoring="roc_auc",
    cv=cv_part3,
    random_state=42,
    n_jobs=-1
)

et_part3_search.fit(X_part3, y)

print("Best Extra Trees Params:", et_part3_search.best_params_)
print("Best Extra Trees ROC AUC:", et_part3_search.best_score_)


Best Extra Trees Params: {'model__n_estimators': 500, 'model__min_samples_split': 5, 'model__min_samples_leaf': 1, 'model__max_features': 'sqrt', 'model__max_depth': 5}
Best Extra Trees ROC AUC: 0.7954010836527026


**Gradient Boosting Model:**

In [52]:
gb_part3_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor_part3),
    ("model", GradientBoostingClassifier(random_state=42))
])

gb_part3_param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "model__max_depth": [2, 3, 4],
    "model__subsample": [0.8, 1.0],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 5]
}

gb_part3_search = RandomizedSearchCV(
    estimator=gb_part3_pipe,
    param_distributions=gb_part3_param_grid,
    n_iter=20,
    scoring="roc_auc",
    cv=cv_part3,
    random_state=42,
    n_jobs=-1
)

gb_part3_search.fit(X_part3, y)

print("Best Gradient Boosting Params:", gb_part3_search.best_params_)
print("Best Gradient Boosting ROC AUC:", gb_part3_search.best_score_)


Best Gradient Boosting Params: {'model__subsample': 1.0, 'model__n_estimators': 100, 'model__min_samples_split': 2, 'model__min_samples_leaf': 2, 'model__max_depth': 2, 'model__learning_rate': 0.05}
Best Gradient Boosting ROC AUC: 0.7948491515014886


**XGBoost Model:**

In [53]:
from xgboost import XGBClassifier

xgb_part3_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor_part3),
    ("model", XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        tree_method="hist"
    ))
])

xgb_part3_param_grid = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [3, 4, 5, 6],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__subsample": [0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.8, 0.9, 1.0],
    "model__min_child_weight": [1, 3, 5],
    "model__gamma": [0, 0.1, 0.3]
}

xgb_part3_search = RandomizedSearchCV(
    estimator=xgb_part3_pipe,
    param_distributions=xgb_part3_param_grid,
    n_iter=20,
    scoring="roc_auc",
    cv=cv_part3,
    random_state=42,
    n_jobs=-1
)

xgb_part3_search.fit(X_part3, y)

print("Best XGBoost Params:", xgb_part3_search.best_params_)
print("Best XGBoost ROC AUC:", xgb_part3_search.best_score_)


Best XGBoost Params: {'model__subsample': 0.8, 'model__n_estimators': 500, 'model__min_child_weight': 1, 'model__max_depth': 3, 'model__learning_rate': 0.01, 'model__gamma': 0.1, 'model__colsample_bytree': 0.8}
Best XGBoost ROC AUC: 0.7939240821212851


**Model Comparisons:**

In [54]:
part3_model_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "Extra Trees",
        "Gradient Boosting",
        "XGBoost"
    ],
    "Best_ROC_AUC": [
        log_part3_search.best_score_,
        tree_part3_search.best_score_,
        rf_part3_search.best_score_,
        et_part3_search.best_score_,
        gb_part3_search.best_score_,
        xgb_part3_search.best_score_
    ]
})

part3_model_results = part3_model_results.sort_values(
    by="Best_ROC_AUC", ascending=False
).reset_index(drop=True)

print(part3_model_results)
print("\nBest Part 3 Model:", part3_model_results.loc[0, "Model"])



                 Model  Best_ROC_AUC
0  Logistic Regression      0.796623
1          Extra Trees      0.795401
2    Gradient Boosting      0.794849
3              XGBoost      0.793924
4        Random Forest      0.789551
5        Decision Tree      0.780955

Best Part 3 Model: Logistic Regression


As of right now the logistic regression model seems to be the best, now we can tune it to make it better considering a ROC AUC score of 79.7% is not the best. 

In [55]:
log_part3_pipe_deep = Pipeline(steps=[
    ("preprocessor", preprocessor_part3),
    ("model", LogisticRegression(max_iter=5000, random_state=42))
])

log_part3_param_grid_deep = [
    {
        "model__solver": ["lbfgs"],
        "model__penalty": ["l2"],
        "model__C": np.logspace(-3, 3, 13),
        "model__class_weight": [None, "balanced"]
    },
    {
        "model__solver": ["liblinear"],
        "model__penalty": ["l1", "l2"],
        "model__C": np.logspace(-3, 3, 13),
        "model__class_weight": [None, "balanced"]
    },
    {
        "model__solver": ["saga"],
        "model__penalty": ["l1", "l2"],
        "model__C": np.logspace(-3, 3, 13),
        "model__class_weight": [None, "balanced"]
    },
    {
        "model__solver": ["saga"],
        "model__penalty": ["elasticnet"],
        "model__C": np.logspace(-3, 3, 13),
        "model__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],
        "model__class_weight": [None, "balanced"]
    }
]

log_part3_search_deep = RandomizedSearchCV(
    estimator=log_part3_pipe_deep,
    param_distributions=log_part3_param_grid_deep,
    n_iter=30,
    scoring="roc_auc",
    cv=cv_part3,
    random_state=42,
    n_jobs=-1
)

log_part3_search_deep.fit(X_part3, y)

print("Best Deep Logistic Params:", log_part3_search_deep.best_params_)
print("Best Deep Logistic ROC AUC:", log_part3_search_deep.best_score_)


Best Deep Logistic Params: {'model__solver': 'saga', 'model__penalty': 'elasticnet', 'model__l1_ratio': 0.1, 'model__class_weight': None, 'model__C': np.float64(1.0)}
Best Deep Logistic ROC AUC: 0.7967288302925241


In [56]:
print("Current Logistic ROC AUC:", log_part3_search.best_score_)
print("Deep-Tuned Logistic ROC AUC:", log_part3_search_deep.best_score_)


Current Logistic ROC AUC: 0.7966231080776437
Deep-Tuned Logistic ROC AUC: 0.7967288302925241


Right now the Deep-Tuned Logistic Regression is the best based off ROC AUC score, but I want to find something better. To do this we will be experimenting with more features.

In [57]:
# Current best base dataset for feature experimentation
X_lr_best = X_part3.copy()
X_submit_lr_best = X_submit_part3.copy()

def add_lr_feature(df, feature_name):
    df = df.copy()

    if feature_name == "PromoCategoryMissing":
        df["PromoCategoryMissing"] = df["PromoCategory"].isna().astype(int)

    elif feature_name == "MonthToMonth_HighCharges":
        high_charge_cutoff = df["MonthlyCharges"].median()
        df["MonthToMonth_HighCharges"] = (
            (df["Contract"] == "Month-to-month") &
            (df["MonthlyCharges"] > high_charge_cutoff)
        ).astype(int)

    elif feature_name == "Fiber_HighCharges":
        high_charge_cutoff = df["MonthlyCharges"].median()
        df["Fiber_HighCharges"] = (
            (df["InternetService"] == "Fiber optic") &
            (df["MonthlyCharges"] > high_charge_cutoff)
        ).astype(int)

    elif feature_name == "Fiber_MonthlyCharges":
        df["Fiber_MonthlyCharges"] = (
            (df["InternetService"] == "Fiber optic").astype(int) * df["MonthlyCharges"]
        )

    return df

lr_feature_candidates = [
    "PromoCategoryMissing",
    "MonthToMonth_HighCharges",
    "Fiber_HighCharges",
    "Fiber_MonthlyCharges"
]

lr_feature_results = []

lr_feature_results.append({
    "Version": "Current Best Dataset",
    "ROC_AUC": log_part3_search_deep.best_score_
})

for feat in lr_feature_candidates:
    X_temp = add_lr_feature(X_lr_best, feat)

    # rebuild column lists and preprocessor for the new feature set
    numeric_features_temp = X_temp.select_dtypes(include=["int64", "float64"]).columns
    categorical_features_temp = X_temp.select_dtypes(include=["object", "category"]).columns

    numeric_transformer_temp = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer_temp = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor_temp = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer_temp, numeric_features_temp),
            ("cat", categorical_transformer_temp, categorical_features_temp)
        ]
    )

    temp_pipe = Pipeline(steps=[
        ("preprocessor", preprocessor_temp),
        ("model", LogisticRegression(max_iter=5000, random_state=42))
    ])

    temp_search = RandomizedSearchCV(
        estimator=temp_pipe,
        param_distributions=log_part3_param_grid_deep,
        n_iter=30,
        scoring="roc_auc",
        cv=cv_part3,
        random_state=42,
        n_jobs=-1
    )

    temp_search.fit(X_temp, y)

    lr_feature_results.append({
        "Version": f"Current Best + {feat}",
        "ROC_AUC": temp_search.best_score_
    })

lr_feature_results_df = pd.DataFrame(lr_feature_results).sort_values(
    by="ROC_AUC", ascending=False
)

print(lr_feature_results_df)

                                   Version   ROC_AUC
1      Current Best + PromoCategoryMissing  0.797068
0                     Current Best Dataset  0.796729
2  Current Best + MonthToMonth_HighCharges  0.796562
3         Current Best + Fiber_HighCharges  0.796559
4      Current Best + Fiber_MonthlyCharges  0.796314


Based on these results we will now add PromoCategoryMissing as a new feature to our dataset. 

In [58]:
X_part3_best = X_part3.copy()
X_submit_part3_best = X_submit_part3.copy()

X_part3_best["PromoCategoryMissing"] = X_part3_best["PromoCategory"].isna().astype(int)
X_submit_part3_best["PromoCategoryMissing"] = X_submit_part3_best["PromoCategory"].isna().astype(int)

In [59]:
numeric_features_best = X_part3_best.select_dtypes(include=["int64", "float64"]).columns
categorical_features_best = X_part3_best.select_dtypes(include=["object", "category"]).columns

numeric_transformer_best = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_transformer_best = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_part3_best = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_best, numeric_features_best),
        ("cat", categorical_transformer_best, categorical_features_best)
    ]
)

log_part3_pipe_final = Pipeline(steps=[
    ("preprocessor", preprocessor_part3_best),
    ("model", LogisticRegression(max_iter=5000, random_state=42))
])

log_part3_search_final = RandomizedSearchCV(
    estimator=log_part3_pipe_final,
    param_distributions=log_part3_param_grid_deep,
    n_iter=30,
    scoring="roc_auc",
    cv=cv_part3,
    random_state=42,
    n_jobs=-1
)

log_part3_search_final.fit(X_part3_best, y)

print("Final Logistic Params:", log_part3_search_final.best_params_)
print("Final Logistic ROC AUC:", log_part3_search_final.best_score_)


Final Logistic Params: {'model__solver': 'saga', 'model__penalty': 'elasticnet', 'model__l1_ratio': 0.9, 'model__class_weight': None, 'model__C': np.float64(3.1622776601683795)}
Final Logistic ROC AUC: 0.797067763275523


**Best Model:**

In [60]:
print("Final Logistic Params:", log_part3_search_final.best_params_)
print("Final Logistic ROC AUC:", log_part3_search_final.best_score_)

Final Logistic Params: {'model__solver': 'saga', 'model__penalty': 'elasticnet', 'model__l1_ratio': 0.9, 'model__class_weight': None, 'model__C': np.float64(3.1622776601683795)}
Final Logistic ROC AUC: 0.797067763275523


### **Prediction:**

In [61]:
# Fit the best model on the full refined training data
final_model = log_part3_search_final.best_estimator_
final_model.fit(X_part3_best, y)

# Generate probability predictions for the 1,000 customers in test_features.csv
pred_probs = final_model.predict_proba(X_submit_part3_best)[:, 1]

print("Number of predictions:", len(pred_probs))
print(pred_probs[:10])

Number of predictions: 1000
[0.21396008 0.01429685 0.36800717 0.00850775 0.04668492 0.17621026
 0.04977855 0.26221823 0.09575643 0.19789159]


### **Creating predictions.csv:**

In [62]:
submission = pd.DataFrame({
    "id": range(1, 1001),
    "prediction": pred_probs
})

print(submission.shape)
print(submission.head())

submission.to_csv("predictions.csv", index=False)
print("predictions.csv saved successfully.")

(1000, 2)
   id  prediction
0   1    0.213960
1   2    0.014297
2   3    0.368007
3   4    0.008508
4   5    0.046685
predictions.csv saved successfully.
